# HW 3: Continuous Heterogeneity

Re-estimate the multinomial logit brand choice model from last week using a continuous heterogeneity distribution and the simulated maximum likelihood approach discussed today.

Please use the attached set of simulated normal draws for your random coefficients:
- M=100 and since N=100, there are 10k draws. 
- Use a unique set of 100 for each individual
- Each m draw is a 1 by 7 vector to represent the 4 intercepts and 3 slope coefficients

In [2]:
# Packages
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import logsumexp as jax_logsumexp

In [3]:
# Import data
N = 100
S = 100

eta_raw = np.loadtxt("../data/eta.txt")
eta = eta_raw.reshape(N, S, 7, order="C")  # first 100 rows are for the first household, etc.
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
df.head()

,hh,qty,exp,nopurch,b1,b2,b3,b4,p1,p2,p3,p4,f1,f2,f3,f4,tripnum
0,1,2,40.900002,0,0,0,0,1,0.11,0.08,0.06,0.08,0,0,0,0,1
1,1,0,8.830000,1,0,0,0,0,0.11,0.09,0.05,0.08,0,0,0,0,2
2,1,0,3.880000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,3
3,1,0,0.780000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,4
4,1,0,39.240002,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,5


In [4]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)

# Create prev_purch variable
df_long["brand_num"] = df_long[["brand_1", "brand_2", "brand_3", "brand_4"]].idxmax(axis=1).str.replace("brand_", "").astype(int)
trip_choice = (
    df_long.loc[df_long["b"] == 1, ["hh", "tripnum", "brand_num"]]
    .rename(columns={"brand_num": "chosen_brand"})
)
trip_hist = (
    df_long[["hh", "tripnum"]]
    .drop_duplicates()
    .sort_values(["hh", "tripnum"])
    .merge(trip_choice, on=["hh", "tripnum"], how="left")
)
trip_hist["prev_purch"] = (
    trip_hist.groupby("hh")["chosen_brand"]
    .transform(lambda x: x.ffill().shift(1))
    .fillna(0)
    .astype(int)
)
df_long = df_long.merge(
    trip_hist[["hh", "tripnum", "prev_purch"]],
    on=["hh", "tripnum"],
    how="left",
)
df_long["prev_purch"] = (df_long["prev_purch"] == df_long["brand_num"]).astype(int)

df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4,brand_num,prev_purch
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0,1,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0,2,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0,3,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1,4,0
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0,1,0


## Random coefficients implementation

<div style="display:none">
$\newcommand{\bm}[1]{\mathbf{#1}}$
</div>

Suppose there are $N$ households $i=1, \ldots, N$, each of whom makes $T_i$ trips and decides between $J$ products or an outside option. For simplicity, we drop the subscript on $T_i$. In total, there are $NT$ choice occasions.

In the random coefficient model, the set of coefficients are household-specific and distributed according to a continuous parametric distribution $g(\bm{\theta})$. For example, $g$ could be the normal distribution parameterized by $\bm{\theta} = (\bm{\bar\beta}, \bm\Sigma)$:
$$
\bm{\beta}_i 
\coloneqq \begin{bmatrix} \beta_{1i} \\ \beta_{2i} \\ \vdots \\ \beta_{ki} \end{bmatrix} 
\sim \mathcal{N}\left( 
    \begin{bmatrix} 
    \bar\beta_{1} \\ \bar\beta_{2} \\ \vdots \\ \bar\beta_{k} 
    \end{bmatrix}, 
    \begin{bmatrix} 
    \sigma_1^2 & \sigma_{1,2} & \cdots & \sigma_{1,k} \\
    \sigma_{2,1} & \sigma_2^2 & \cdots & \sigma_{2,k} \\
    \vdots & \vdots & \ddots & \vdots \\
    \sigma_{k, 1} & \sigma_{k, 2} & \cdots & \sigma_k^2
    \end{bmatrix} 
    \right)
\eqqcolon \mathcal{N}(\bm{\bar\beta}, \bm\Sigma)
$$
Equivalently, letting $\bm{\Gamma} \in \mathbb{R}^{k \times k}$ be the Cholesky decomposition of $\bm\Sigma$ (i.e., $\bm\Sigma = \bm{\Gamma}\bm{\Gamma}^\intercal$), we have
$$
\bm{\beta}_{i} = \bm{\bar\beta} + \bm\Gamma \eta_i, \quad \eta_i \sim \mathcal{N}(0, I_{k \times k})
$$
Note that $\bm{\Gamma}$ is lower-triangular, so there are only $\frac{k(k+1)}{2}$ parameters to estimate to identify the covariance matrix. If there is no perfect multicollinearity among the brand intercepts, then $\bm{\Sigma}$ is positive definite and the Cholesky factorization $\bm{\Gamma}$ is unique.

The likelihood of household $i$'s observed choices $\bm{y}_i = (y_{i1}, \ldots, y_{iT})$ given their covariates $\bm{x}_i = (x_{i1}, \ldots, x_{iT})$ and the distribution parameters $\bm\theta$ is
$$\begin{align*}
L_i 
&= L(\bm{y}_i \mid \bm{x}_i, \bm\theta) \\
&= \int \prod_{t=1}^T L(y_{it} \mid x_{it}, \bm\beta) g(\bm\beta \mid \bm\theta) \mathrm{d} \bm\beta \\
&= \int \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}, \bm{\Gamma}, \eta_i) \phi(\eta_i) \mathrm{d} \eta_i
\end{align*}$$
where $\phi(\cdot)$ is the pdf of the standard normal. To simulate the integral, we take $S$ draws of $\eta_i$ (for each $i=1, \ldots, N$) according to $\phi(\cdot)$. Call these draws $\eta_i^{(1)}, \ldots, \eta_i^{(S)}$. The likelihood can be approximated as
$$
\tilde{L}_i 
= \frac{1}{S} \sum_{s=1}^S L_i(\bm{y}_i \mid \bm{x}_i, \bar{\bm\beta}, \bm{\Gamma}, \eta_i^{(s)})
= \frac{1}{S} \sum_{s=1}^S \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}, \bm{\Gamma}, \eta_i^{(s)})
$$
To compute this likelihood, we use a similar method as described in `2-discrete-heterogeneity.ipynb`. At a high level, the key difference is that we replace classes with the $S$ draws and sum over the $S$ draws rather than take a class-proportion weighted sum. Let $\bm{H}_N \in \mathrm{R}^{N \times S \times k}$ be a matrix of random draws from the standard normal and let $\bar{\bm{B}} \in \mathbb{R}^{N \times S \times k}$ be $\bar{\bm\beta}$ copied $S \times N$ times in the conforming directions. Then, the simulated coefficients for all households for all draws are contained in
$$
\bm{B} = \bar{\bm{B}} + \bm{H}_N \bm{\Gamma}^\intercal \in \mathbb{R}^{N \times S \times k}
$$
where $\bm{H}_N \bm{\Gamma}^\intercal$ is batched on the first dimension such that it evalutes $N$ separate matrix multiplications between $\bm{H}_N[i, :, :] \in \mathbb{R}^{S \times k}$ and $\bm{\Gamma}^\intercal \in \mathbb{R}^{k \times k}$. Now create $\bm{B}' \in \mathbb{R}^{NTJ \times S \times k}$ that repeats each household $i$'s simulated coefficients $T_iJ$ times in the first dimension. We create this object to match the dimensions of the design matrix $\bm{X} \in \mathbb{R}^{NTJ \times k}$. Then, $\bm{V} \in \mathbb{R}^{NTJ \times S}$ defined by 
$$
\bm{V}_{is} = \sum_{\ell=1}^k \bm{B}'_{is\ell} X_{i\ell} 
$$
contains the indirect utility $v_{itj \mid s}$ of household $i$ in trip $t$ choosing alternative $j$, if their coefficient vector is $\bm{\beta}_i^{(s)} = \bm{\bar\beta} + \bm\Gamma \eta_i^{(s)}$. The rest of the workflow follows similarly to the one described in `2-discrete-heterogeneity.ipynb`, so refer to that notebook for details. The total log-likelihood for the dataset is given by
$$\begin{align*}
\sum_{i=1}^N \log(\tilde{L}_i) 
&= \sum_{i=1}^N \log\left(\frac{1}{S} \sum_{s=1}^S \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}_0, \bm{\Gamma}, \bm{\beta}_{\mathrm{com}}, \eta^{(s)}\right) \\
&= \sum_{i=1}^N \left( -\log(S) + \log\left(\sum_{s=1}^S \exp\left( \log\left( \prod_{t=1}^T L(y_{it} \mid x_{it}, \bar{\bm\beta}_0, \bm{\Gamma}, \bm{\beta}_{\mathrm{com}}, \eta^{(s)}\right)\right)\right) \right) \\
% &= \sum_{i=1}^N \left( -\log(S) + \log\left(\sum_{s=1}^S \exp\left( \log\left( \prod_{t=1}^T \frac{\exp(x_{itj}\bm\beta_{j_{it}^*}^{(s)})}{1+\sum_{j=1}^J \exp(x_{itj}\bm\beta_{j}^{(s)})} \right)\right)\right) \right) \\
&= \sum_{i=1}^N \left( -\log(S) + \mathrm{logsumexp}_s \left( \sum_{t=1}^T \left( \bm{V}_\text{chosen} - \mathrm{logsumexp}_j([\mathbf{0}, \bm{V}]) \right)\right) \right) \\
\end{align*}$$

### Aside: Allowing for common coefficients via restrictions on covariance matrix

<div style="display:none">
$\newcommand{\bm}[1]{\mathbf{#1}}$
</div>

Suppose we don't want all $k$ coefficients to be household-specific. For example, we may only want brand coefficients $\bm{\bm\beta}_{0i}$ to be heterogeneous while other coefficients $\beta_1, \ldots, \beta_{\tilde{k}}$ are shared by all households. In this case, we can express the distribution of the covariate vector as
$$
\bm{\beta}_i = \begin{bmatrix} \bm{\bm\beta}_{0i} \\ \beta_1 \\ \vdots \\ \beta_{\tilde{k}} \end{bmatrix}
\sim \mathcal{N}\left( 
    \begin{bmatrix} 
    \bm{\bar\beta_0} \\ \beta_1 \\ \vdots \\ \beta_{\tilde{k}} 
    \end{bmatrix}, 
    \begin{bmatrix} 
    \bm\Sigma & \bm{0}_{J \times \tilde{k}} \\
    \bm{0}_{\tilde{k} \times J} & \bm{0}_{\tilde{k} \times \tilde{k}}
    \end{bmatrix} 
    \right)
\eqqcolon \mathcal{N}(\bm{\bar\beta}, \tilde{\bm\Sigma})
$$
That is, we restrict variances and covariances corresponding to common coefficients to be 0. The corresponding Cholesky decomposition form is
$$
\bm{\beta}_i = \bm{\bar\beta} + \bm\Gamma \eta_i, \quad \eta_i \sim \mathcal{N}(0, I_{J \times J}) 
$$
where
$$
\bm\Gamma = 
\begin{bmatrix} 
    \tilde{\bm\Gamma} & \bm{0}_{J \times \tilde{k}} \\
    \bm{0}_{\tilde{k} \times J} & \bm{0}_{\tilde{k} \times \tilde{k}}
\end{bmatrix}, \quad \tilde{\bm\Gamma}\tilde{\bm\Gamma}^\intercal = \bm\Sigma
$$
The parameters to be estimated are the $k$ coefficients in $\bm{\bar\beta}$ and the $\frac{J(J+1)}{2}$ parameters in $\tilde{\bm{\Gamma}}$.

In [5]:
def make_gamma_mask(k_het, structure='full', blocks=None):
    """
    Build a lower-triangular boolean mask for the Cholesky factor Gamma_tilde.

    Parameters
    ----------
    k_het : int
        Size of the heterogeneous block (number of random coefficients).
    structure : str
        'full'     — all lower-triangular entries free (default; full covariance)
        'diagonal' — only diagonal entries free (diagonal covariance, no correlations)
        'block'    — block-diagonal; ``blocks`` is a list of group sizes summing to k_het
    blocks : list of int or None
        Required when structure='block'. Group sizes, e.g. [4, 3] for a 7x7 matrix
        split into a 4x4 block and a 3x3 block.

    Returns
    -------
    mask : np.ndarray of bool, shape (k_het, k_het), lower triangular
    """
    mask = np.zeros((k_het, k_het), dtype=bool)
    if structure == 'full':
        mask[np.tril_indices(k_het)] = True
    elif structure == 'diagonal':
        np.fill_diagonal(mask, True)
    elif structure == 'block':
        if blocks is None:
            raise ValueError("blocks must be provided for structure='block'")
        if sum(blocks) != k_het:
            raise ValueError(f"blocks must sum to k_het={k_het}; got sum={sum(blocks)}")
        idx = 0
        for sz in blocks:
            rows, cols = np.tril_indices(sz)
            mask[rows + idx, cols + idx] = True
            idx += sz
    else:
        raise ValueError(f"Unknown structure '{structure}'. Use 'full', 'diagonal', or 'block'.")
    return mask


def _panel_indices(hh_ids, n_alts, n_rows):
    """Map each row of the long panel to a household index in 0..N-1."""
    hh_ids = np.asarray(hh_ids).reshape(-1)
    n_choices = n_rows // n_alts
    hh_trip = hh_ids.reshape(n_choices, n_alts, order="C")[:, 0]
    _, hh_index = np.unique(hh_trip, return_inverse=True)
    return hh_index


def _unpack_theta(theta, k, k_het, gamma_mask=None, final_out=False):
    """
    Unpack theta into mean coefficients (k,) and Cholesky factor for heterogeneous coefficients.

    Theta ordering is [mean coefficients (length k), free lower-triangular elements of Γ̃
    (length = gamma_mask.sum(), or k_het*(k_het+1)//2 if gamma_mask is None)].

    gamma_mask : np.ndarray of bool, shape (k_het, k_het), lower triangular, or None.
        Specifies which entries of Γ̃ are free parameters. None means all lower-triangular
        entries are free (full covariance). Use make_gamma_mask() to build common structures.

    Γ is k × k_het with Γ̃ in the top k_het rows (lower triangular) and zeros below, so that
    β = β̄ + Γ η with η ~ N(0, I_{k_het}) and common coefficients have zero variance.
    Columns of X must be ordered with heterogeneous covariates first, then common (see estimate_MNL).
    """
    betas = theta[:k]

    if gamma_mask is None:
        mask_i, mask_j = np.tril_indices(k_het)
    else:
        mask_i, mask_j = np.where(gamma_mask)

    p = len(mask_i)
    gamma_flat = theta[k : k + p]
    is_diag = (mask_i == mask_j)
    gamma_vals = jnp.where(is_diag, jnp.exp(gamma_flat), gamma_flat)  # exp enforces positive diagonal

    Gamma_tilde = jnp.zeros((k_het, k_het)).at[mask_i, mask_j].set(gamma_vals)

    if k_het == k or final_out:
        Gamma = Gamma_tilde
    else:
        Gamma = jnp.concatenate(
            [Gamma_tilde, jnp.zeros((k - k_het, k_het))],
            axis=0,
        )
    return betas, Gamma


def _negloglik_hh(theta, X, Y, eta, hh_index, n_alts, k_het, gamma_mask=None):
    """
    Negative log-likelihoods for random coefficient MNL with outside option.
    Returns (N,) — one value per household.
    """
    N = int(np.max(np.asarray(hh_index))) + 1
    k = X.shape[1]
    n_choices = Y.shape[0] // n_alts

    theta = jnp.asarray(theta, dtype=jnp.float64).reshape(-1)
    X = jnp.asarray(X, dtype=jnp.float64)
    Y = jnp.asarray(Y, dtype=jnp.float64).reshape(-1)
    eta = jnp.asarray(eta, dtype=jnp.float64)
    hh_index = jnp.asarray(hh_index, dtype=jnp.int32)

    n_draws = eta.shape[1]                                              # S
    mean_beta, Gamma = _unpack_theta(theta, k, k_het, gamma_mask)       # (k,) and (k, k_het)
    B = mean_beta[None, None, :] + eta @ Gamma.T                        # (N, S, k)
    hh_index1 = jnp.repeat(hh_index, n_alts)
    B_hh = B[hh_index1, :, :]                                           # (NTJ, S, k)

    V = jnp.einsum("ij,isj->is", X, B_hh)                               # (NTJ, S)
    V = V.reshape(n_choices, n_alts, n_draws, order="C")                # (NT, J, S)
    V0 = jnp.concatenate([jnp.zeros((n_choices, 1, n_draws)), V], axis=1)
    Y1 = Y.reshape(n_choices, n_alts, 1, order="C")                     # (NT, J, 1)
    chosen_v = jnp.multiply(Y1, V).sum(axis=1)                          # (NT, S)
    log_L3 = chosen_v - jax_logsumexp(V0, axis=1)                       # (NT, S)

    log_L2 = jax.ops.segment_sum(log_L3, hh_index, num_segments=N)      # (N_hh, S)
    log_L1 = -jnp.log(jnp.asarray(n_draws, dtype=theta.dtype)) + jax_logsumexp(log_L2, axis=1)  # (N,)
    return -log_L1


def _negloglik(theta, X, Y, eta, hh_index, n_alts, k_het, gamma_mask=None):
    """Summed negative log-likelihood."""
    negLL = _negloglik_hh(theta, X, Y, eta, hh_index, n_alts, k_het, gamma_mask)
    return np.sum(negLL)


def _score_matrix(theta, X, Y, eta, hh_index, n_alts, k_het, gamma_mask=None):
    """
    Score matrix of household-level negative log-likelihood: d (NLL_h) / d theta.
    Shape (n_households, n_params). Computed with JAX reverse-mode Jacobian.
    """
    def neglog_hh(th):
        return _negloglik_hh(th, X, Y, eta, hh_index, n_alts, k_het, gamma_mask)

    J = jax.jacrev(neglog_hh)(theta)
    return np.asarray(J)


def _print_mle_output(output, df, n_alts, covariate_vars, robust_se, individual_var, k_het):
    """Prints MLE results in a nice table."""
    
    B_hat = np.asarray(output['opt_beta'], dtype=float).reshape(-1)
    Sigma_hat = np.asarray(output['opt_sigma'])
    
    n_choices = df.shape[0] // n_alts               # NT
    n_households = df[individual_var].nunique()     # N
    k = B_hat.shape[0]

    se_theta = np.asarray(output['se_theta'], dtype=float).reshape(-1)
    ci_theta = np.asarray(output['ci_theta'], dtype=float)
    if ci_theta.ndim == 1:
        ci_theta = ci_theta.reshape(-1, 2)

    se_beta = np.zeros(k, dtype=float)
    ci_beta = np.zeros((k, 2), dtype=float)

    se_beta[:k] = se_theta[:k]
    ci_beta[:k, :] = ci_theta[:k, :]

    print(output.get('message', ''))
    print(f"Converged in {output['num_iter']} iterations.")
    print()
    print(f"Log-likelihood: {output['opt_ll']:.4f}")
    print(f"Number of individuals: {n_households}")
    print(f"Number of choice occasions: {n_choices}")
    print(f"Number of parameters estimated: {output['opt_theta'].shape[0]}")
    if robust_se:
        print("Robust standard errors, clustered on the", individual_var, "variable.")
    else:
        print("Standard errors are not robust and assume correct model specification.")
    print()

    widths = (12, 10, 11, 10, 10)
    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'[Confidence Interval]':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)

    print("Population Mean of Coefficients")
    print(header_top)
    print(divider)
    for i, name in enumerate(covariate_vars):
        row = " | ".join([
            f"{name:<{widths[0]}}",
            f"{B_hat[i]:>{widths[1]}.6f}",
            f"{se_beta[i]:>{widths[2]}.6f}",
            f"{ci_beta[i, 0]:>{widths[3]}.5f}",
            f"{ci_beta[i, 1]:>{widths[4]}.5f}"
        ])
        print(row)
    print()

    print("Population Covariance Matrix")
    cov_print = pd.DataFrame(
        np.round(Sigma_hat, 4), 
        index=covariate_vars[:k_het], 
        columns=covariate_vars[:k_het])
    upper = np.triu(np.ones(cov_print.shape, dtype=bool), k=1)
    print(cov_print.mask(upper).to_string(na_rep=""))

def _mle_estimator(df,
                   choice_var,
                   covariate_vars,
                   hh_index,
                   n_alts,
                   eta,
                   beta_init,
                   gamma_init,
                   k_het,
                   ci_alpha,
                   robust_se,
                   opt_method,
                   gamma_mask=None):
    """
    Estimates the multinomial logit model with random coefficients using maximum likelihood estimation,
    from initial values (beta_init, gamma_init).

    k_het : int
        Number of heterogeneous coefficients (columns of X that load on η). Must match eta.shape[2].
    gamma_mask : np.ndarray of bool, shape (k_het, k_het), lower triangular, or None.
        Which entries of Gamma_tilde are free parameters. None means full lower triangle.
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    k = X.shape[1]

    # Analytic gradient (JAX reverse-mode); close over gamma_mask so it's treated as static
    _X_j = jnp.asarray(X, dtype=jnp.float64)
    _Y_j = jnp.asarray(Y, dtype=jnp.float64).reshape(-1)
    _eta_j = jnp.asarray(eta, dtype=jnp.float64)
    _hh_j = jnp.asarray(hh_index, dtype=jnp.int32)

    def _nll_sum_jax(th):
        return jnp.sum(
            _negloglik_hh(th, _X_j, _Y_j, _eta_j, _hh_j, n_alts, k_het, gamma_mask)
        )

    # JIT-compiled gradient and Hessian of neg LL w.r.t. theta
    _grad_nll_jit = jax.jit(jax.grad(_nll_sum_jax))
    _hess_nll_jit = jax.jit(jax.hessian(_nll_sum_jax))

    def _jac_negloglik(theta, *args):
        g = _grad_nll_jit(jnp.asarray(theta, dtype=jnp.float64).reshape(-1))
        return np.asarray(g, dtype=float)

    theta_init = np.concatenate([beta_init, gamma_init])

    # Run optimization
    result = minimize(
        fun=_negloglik,
        jac=_jac_negloglik,
        x0=theta_init,
        args=(X, Y, eta, hh_index, n_alts, k_het, gamma_mask),
        method=opt_method
    )

    # Extract results
    opt_ll = -result.fun
    opt_theta = result.x
    B_hat, Gamma_hat = _unpack_theta(opt_theta, k, k_het, gamma_mask, final_out=True)
    Sigma_hat = Gamma_hat @ Gamma_hat.T

    # Compute standard errors and confidence intervals using the analytic Hessian
    H = np.asarray(_hess_nll_jit(jnp.asarray(opt_theta, dtype=jnp.float64)))
    H_inv = np.linalg.pinv(H)
    se_theta = np.sqrt(np.clip(np.diag(H_inv), 0.0, None))  # avoid negative variances from numerical issues

    z_score = norm.ppf(1 - ci_alpha / 2)
    ci_theta = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta)]

    # Collect output
    output = {
        'opt_ll': opt_ll,
        'opt_theta': opt_theta,
        'opt_beta': B_hat,
        'opt_gamma': Gamma_hat,
        'opt_sigma': Sigma_hat,
        'se_theta': se_theta,
        'ci_theta': ci_theta,
        'success': result.success,
        'message': result.message,
        'num_iter': result.nit,
    }

    if robust_se:
        # J estimated by outer products of household-level scores.
        S = _score_matrix(opt_theta, X, Y, eta, hh_index, n_alts, k_het, gamma_mask)
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_theta_r = np.sqrt(np.clip(np.diag(V_robust), 0.0, None))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta_r)]

        output['se_theta'] = se_theta_r
        output['ci_theta'] = ci_r
        output['param_cov'] = V_robust
    else:
        output['param_cov'] = H_inv

    return output


def _uniform_around(rng, center, n_draws, rel_width, abs_floor):
    """Draw shape (n_draws, len(center)) with each coordinate j uniform on
    [c_j - rel_width * s_j, c_j + rel_width * s_j], s_j = max(|c_j|, abs_floor)."""
    c = np.asarray(center, dtype=float).reshape(-1)
    s = np.maximum(np.abs(c), abs_floor)
    low = c - rel_width * s
    high = c + rel_width * s
    return rng.uniform(low=low, high=high, size=(n_draws, c.size))


def _print_start_progress(done, total, t0, current_ll, best_ll):
    """Live progress tracker for randomized starts."""
    elapsed = time.time() - t0
    print(
        f"\rRandomized starts completed: {done}/{total} | "
        f"Elapsed {elapsed:,.1f}s | "
        f"Current LL: {current_ll:,.4f} | "
        f"Best LL so far: {best_ll:,.4f}",
        end="",
        flush=True,
    )

def _run_mle_from_start(start_idx, beta0, gamma0, estimator_kwargs):
    t_start = time.time()
    try:
        out = _mle_estimator(beta_init=beta0, gamma_init=gamma0, **estimator_kwargs)
    except Exception as e:
        out = {
            "opt_ll": -np.inf,
            "success": False,
            "message": f"Start failed: {type(e).__name__}: {e}",
            "num_iter": None,
            "opt_theta": None, "opt_beta": None, "opt_gamma": None, "opt_sigma": None,
            "se_theta": None, "ci_theta": None, "param_cov": None,
        }
    out["elapsed_time"] = time.time() - t_start
    return start_idx, out


def estimate_MNL(df,
                 choice_var,
                 covariate_vars,
                 individual_var,
                 n_alts,
                 eta,
                 common_vars=None,
                 beta_init=None,
                 gamma_init=None,
                 sigma_init=None,
                 gamma_mask=None,
                 ci_alpha=0.05,
                 robust_se=False,
                 randomized_starts=0,
                 search_rel_width=3.0,
                 search_abs_floor=1.0,
                 seed=None,
                 n_cores=1,
                 opt_method='BFGS',
                 output_log=False,
                 log_csv_path=None):
    """
    Estimates the multinomial logit model with random coefficients using maximum likelihood estimation.
    
    Optional randomized starts help avoid local optima: for each starting value x, each coordinate is
    drawn uniformly in [x - w*s, x + w*s] with w = ``search_rel_width`` and s = max(|x|, ``search_abs_floor``)
    (so coefficients on different scales get comparable relative moves, and |x|≈0 does not collapse the range).
    Specifying `n_cores` > 1 evaluates these starts in parallel using multiprocessing.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        Column names in estimation order: list every heterogeneous covariate first, then every common
        covariate (fixed coefficient). When ``common_vars`` is non-empty, its entries must match the
        last ``len(common_vars)`` names here, in the same order.
    individual_var : str
        The column name of the individual ID variable.
        If robust standard errors are requested, standard errors are clustered by this variable.
    n_alts : int
        Number of alternatives.
    eta : array
        Random draws from the standard normal. Shape (N, S, k) with columns aligned to
        ``covariate_vars``. Only the first k_het columns (heterogeneous block) are used in simulation.
    common_vars : list or None
        Names of covariates whose coefficients are fixed across households (zero rows/columns in Σ).
        If None (default), all covariates are random. If provided, must equal
        ``covariate_vars[-len(common_vars):]`` (heterogeneous names come first in ``covariate_vars``).
    beta_init : array, optional
        Array of shape (k,) with initial guess for coefficients. 
        If None, defaults to zeros.
    gamma_init : array, optional
        Array of shape (k_het, k_het) with initial guess for Cholesky factor of covariance matrix, 
        where k_het is the number of heterogeneous covariates.
        If None, defaults to identity matrix.
    sigma_init : array, optional
        Array of shape (k_het, k_het) with initial guess for covariance matrix, 
        where k_het is the number of heterogeneous covariates.
        If None, defaults to identity matrix.
    gamma_mask : np.ndarray of bool, shape (k_het, k_het), lower triangular, or None
        Specifies which entries of the Cholesky factor Gamma_tilde are free parameters.
        Entries set to False are fixed at zero, restricting the covariance structure.
        None (default) means all lower-triangular entries are free (full covariance).
        Use make_gamma_mask() to build common structures:
          make_gamma_mask(k_het, 'diagonal')         — no cross-coefficient correlations
          make_gamma_mask(k_het, 'block', [j1, j2])  — block-diagonal covariance
          make_gamma_mask(k_het, 'full')              — full covariance (same as None)
        All diagonal entries must be free (diagonal variances cannot be restricted to zero).
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors. 
        If True, standard errors are clustered by `individual_var`.
    randomized_starts : int
        Total number of optimization starts, including the base start from the
        provided initial values.
    search_rel_width : float
        Multiplicative factor w: each coordinate is perturbed uniformly in
        x ± w * max(|x|, search_abs_floor).
    search_abs_floor : float
        Minimum scale when |x| is zero or tiny, so search intervals do not vanish.
    seed : int or None
        Random seed for reproducibility of randomized starts.
    n_cores : int
        Number of worker processes (CPU cores) for evaluating randomized starts in parallel.
        If 1 (default), evaluates starts serially.
        Use an integer > 1 to opt in to parallel processing.
    opt_method : str
        Optimization method.
    output_log : bool
        If True, include a 'start_log' DataFrame in output showing achieved LL, elapsed time, 
        optimization status and message, starting params, parameter estimates for each start.
    log_csv_path : str, pathlib.Path, or None
        If ``output_log`` is True and this is not None, write the start log DataFrame to this CSV path.

    Returns
    -------
    dict
        Includes ``opt_ll``, ``opt_theta``, ``opt_beta``, ``opt_gamma``, ``opt_sigma``, ``se_theta``,
        ``ci_theta``, ``param_cov`` ((k+p)×(k+p) covariance used for SEs), and optionally ``start_log``.
    """
    k = len(covariate_vars)
    common_vars = [] if common_vars is None else list(common_vars)
    unknown = set(common_vars) - set(covariate_vars)
    if unknown:
        raise ValueError(f"common_vars contains names not in covariate_vars: {unknown}")
    k_common = len(common_vars)
    k_het = k - k_common
    if k_het < 1:
        raise ValueError("Need at least one heterogeneous coefficient (check common_vars).")
    if k_common > 0 and covariate_vars[-k_common:] != common_vars:
        raise ValueError(
            "covariate_vars must list heterogeneous covariates first, then common covariates. "
            "Require covariate_vars[-len(common_vars):] == common_vars (same order)."
        )

    # Build and validate gamma_mask
    if gamma_mask is None:
        gamma_mask_arr = np.tril(np.ones((k_het, k_het), dtype=bool))
    else:
        gamma_mask_arr = np.asarray(gamma_mask, dtype=bool)
        if gamma_mask_arr.shape != (k_het, k_het):
            raise ValueError(f"gamma_mask must have shape ({k_het}, {k_het}); got {gamma_mask_arr.shape}")
        if not np.all(gamma_mask_arr == np.tril(gamma_mask_arr)):
            raise ValueError("gamma_mask must be lower triangular")
        if not np.all(np.diag(gamma_mask_arr)):
            raise ValueError("gamma_mask must include all diagonal entries (variances cannot be fixed to zero)")
    mask_i_np, mask_j_np = np.where(gamma_mask_arr)
    p = len(mask_i_np)

    eta = np.asarray(eta)
    n_hh = df[individual_var].nunique()
    if eta.shape[0] != n_hh or eta.shape[2] != k:
        raise ValueError(f"eta must have shape (N, S, k)=({n_hh}, S, {k}); got {eta.shape}")
    eta_het = eta[:, :, :k_het]

    individual_ids = df[individual_var].to_numpy()
    hh_index = _panel_indices(individual_ids, n_alts, df.shape[0])

    if beta_init is None:
        beta_base = np.zeros(k)
    else:
        beta_base = np.asarray(beta_init).reshape(-1)

    if (gamma_init is None) and (sigma_init is None):
        gamma_mat = np.eye(k_het)
    elif (gamma_init is not None):
        gamma_mat = np.asarray(gamma_init)
        if gamma_mat.shape != (k_het, k_het):
            raise ValueError(f"gamma_init must have shape ({k_het}, {k_het}); got {gamma_mat.shape}")
    elif (sigma_init is not None):
        sigma_mat = np.asarray(sigma_init)
        if sigma_mat.shape != (k_het, k_het):
            raise ValueError(f"sigma_init must have shape ({k_het}, {k_het}); got {sigma_mat.shape}")
        gamma_mat = np.linalg.cholesky(sigma_mat)

    # Log-parameterize diagonal of gamma_mat, then extract only free (masked) entries
    gamma_mat_log = gamma_mat.copy()
    gamma_mat_log[np.diag_indices(k_het)] = np.log(np.diag(gamma_mat))
    gamma_base = gamma_mat_log[mask_i_np, mask_j_np]

    # Generate randomized starts
    rng = np.random.default_rng(seed)
    n_rand = max(int(randomized_starts)-1, 0)
    beta_rand = _uniform_around(rng, beta_base, n_rand, search_rel_width, search_abs_floor)
    gamma_rand = _uniform_around(rng, gamma_base, n_rand, search_rel_width, search_abs_floor)
    
    starts = [(beta_base.copy(), gamma_base.copy())] + list(zip(beta_rand, gamma_rand))

    # Assign starts to workers and run optimizations
    estimator_kwargs = {
        'df': df,
        'choice_var': choice_var,
        'covariate_vars': covariate_vars,
        'hh_index': hh_index,
        'n_alts': n_alts,
        'eta': eta_het,
        'k_het': k_het,
        'ci_alpha': ci_alpha,
        'robust_se': robust_se,
        'opt_method': opt_method,
        'gamma_mask': gamma_mask_arr,
    }

    n_starts = len(starts)
    max_workers = max(1, int(n_cores))
    max_workers = min(max_workers, n_starts)

    best_output = None
    start_log = []
    start_outputs = {}
    done_starts = 0
    t0_starts = time.time()

    if max_workers > 1:
        try:
            print(f"Evaluating {n_starts} randomized starts in parallel using {max_workers} cores...")
            mp_context = multiprocessing.get_context('fork')
            with ProcessPoolExecutor(max_workers=max_workers, mp_context=mp_context) as ex:
                futures = [
                    ex.submit(_run_mle_from_start, i, beta0, gamma0, estimator_kwargs)
                    for i, (beta0, gamma0) in enumerate(starts)
                ]
                for fut in as_completed(futures):
                    start_idx, out = fut.result()
                    if output_log:
                        start_log.append((start_idx, out['opt_ll'], out['elapsed_time'], out['success'], 
                                          out['message'], out['num_iter'], starts[start_idx][0], starts[start_idx][1],
                                          out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                        start_outputs[start_idx] = out
                    if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                        best_output = out
                    done_starts += 1
                    _print_start_progress(done_starts, n_starts, t0_starts, out['opt_ll'], best_output['opt_ll'])
            print()
        except Exception as exc:
            print(f'Parallel starts failed ({exc}); falling back to serial evaluation.')
            for i, (beta0, gamma0) in enumerate(starts):
                _, out = _run_mle_from_start(i, beta0, gamma0, estimator_kwargs)
                if output_log:
                    start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                      out['message'], out['num_iter'], starts[i][0], starts[i][1],
                                      out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                    start_outputs[i] = out
                if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                    best_output = out
                done_starts += 1
                _print_start_progress(done_starts, n_starts, t0_starts, out['opt_ll'], best_output['opt_ll'])
            print()
    else:
        for i, (beta0, gamma0) in enumerate(starts):
            _, out = _run_mle_from_start(i, beta0, gamma0, estimator_kwargs)
            if output_log:
                start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                  out['message'], out['num_iter'], starts[i][0], starts[i][1],
                                  out['opt_beta'], out['opt_gamma'], out['opt_sigma']))
                start_outputs[i] = out
            if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                best_output = out
            done_starts += 1
            _print_start_progress(done_starts, n_starts, t0_starts, out['opt_ll'], best_output['opt_ll'])
        print()

    _print_mle_output(output=best_output, df=df, n_alts=n_alts,
                      covariate_vars=covariate_vars, robust_se=robust_se, 
                      individual_var=individual_var, k_het=k_het)
    
    if output_log:
        log_df = pd.DataFrame(start_log, columns=[
            'start_idx', 'achieved_ll', 'elapsed_time', 'success', 'message', 'num_iter',
            'start_beta', 'start_gamma', 'opt_beta', 'opt_gamma', 'opt_sigma'])
        log_df.sort_values('achieved_ll', ascending=False, inplace=True)
        best_output['start_log'] = log_df.reset_index(drop=True)
        best_output['start_outputs'] = start_outputs
        if log_csv_path is not None:
            path = Path(log_csv_path)
            path.parent.mkdir(parents=True, exist_ok=True)
            log_df.to_csv(path, index=False)
    
    return best_output

### Full heterogeneity (no restrictions)

In [ ]:
# beta_init = [0.5, -1, -4, -2, 0.5, -40, 1.5]
# sigma_init = np.diag([4, 4, 4, 4, 1, 100, 1])

log_fullhet = pd.read_csv("./output_logs/randcoef_all_log24.csv")
beta_init = log_fullhet.iloc[1]['opt_beta']
beta_init = np.fromstring(beta_init.strip('[]'), sep=' ')
gamma_init = log_fullhet.iloc[1]['opt_gamma']
gamma_init = np.fromstring(gamma_init.replace('[', ' ').replace(']', ' '), sep=' ').reshape(7, 7)

results_fullhet = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    eta=eta,
    common_vars=None,
    beta_init=beta_init,
    gamma_init=gamma_init,
    sigma_init=None,
    robust_se=True,
    randomized_starts=1,
    search_rel_width=2,
    search_abs_floor=1,
    opt_method='BFGS',
    output_log=True,
    log_csv_path="output_logs/randcoef_all_log.csv"
)

NameError: name 'pd' is not defined

### Heterogeneity on brand intercepts, no covariance restrictions

In [45]:
# beta_init = [0.5, -0.5, -3.5, -2, 0.5, -40, 1.5]
# sigma_init = np.diag([3, 3, 3, 3])

log_brandhet = pd.read_csv("./output_logs/randcoef_brand_log24.csv")
beta_init = log_brandhet.iloc[0]['opt_beta']
beta_init = np.fromstring(beta_init.strip('[]'), sep=' ')
gamma_init = log_brandhet.iloc[1]['opt_gamma']
gamma_init = np.fromstring(gamma_init.replace('[', ' ').replace(']', ' '), sep=' ').reshape(4, 4)

results_brandhet = estimate_MNL(
    df=df_long,
    choice_var="b",
    covariate_vars=["brand_1", "brand_2", "brand_3", "brand_4", "f", "p", "prev_purch"],
    individual_var="hh",
    n_alts=4,
    eta=eta,
    common_vars=["f", "p", "prev_purch"],
    beta_init=beta_init,
    gamma_init=gamma_init,
    sigma_init=None,
    robust_se=True,
    randomized_starts=1,
    search_rel_width=4,
    search_abs_floor=1,
    opt_method="BFGS",
    output_log=True,
    log_csv_path="output_logs/randcoef_brand_log.csv"
)

Randomized starts completed: 1/1 | Elapsed 46.7s | Current LL: -6,443.0852 | Best LL so far: -6,443.0852
Desired error not necessarily achieved due to precision loss.
Converged in 0 iterations.

Log-likelihood: -6443.0852
Number of individuals: 100
Number of choice occasions: 12784
Number of parameters estimated: 17
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.581813 |    0.419095 |   -0.23960 |    1.40322
brand_2      |  -0.436905 |    0.320698 |   -1.06546 |    0.19165
brand_3      |  -3.706852 |    0.414243 |   -4.51875 |   -2.89495
brand_4      |  -2.418978 |    0.321533 |   -3.04917 |   -1.78878
f            |   0.377075 |    0.136135 |    0.11026 |    0.64389
p            | -39.242875 |    3.717789 |  -46.52961 |  -31.95614
prev_purch   |   1.222841 |    0.121121 |    0.98545 |    1.46023



### Heterogeneity on all coefficients, diagonal covariance

In [ ]:
beta_init = [0.5, -1, -4, -2, 0.5, -40, 1.5]
sigma_init = np.diag([2, 2, 2, 2, 0.5, 40, 0.5])

results_allcoef_diagcov = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    eta=eta,
    common_vars=None,
    beta_init=beta_init,
    sigma_init=sigma_init,
    gamma_mask=make_gamma_mask(7, 'diagonal'),
    robust_se=True,
    randomized_starts=1,
    search_rel_width=2,
    search_abs_floor=1,
    opt_method='BFGS',
    output_log=True,
    log_csv_path="output_logs/randcoef_allcoef_diagcov_log.csv"
)

Randomized starts completed: 1/1 | Elapsed 36.8s | Current LL: -6,453.8924 | Best LL so far: -6,453.8924
Optimization terminated successfully.
Converged in 34 iterations.

Log-likelihood: -6453.8924
Number of individuals: 100
Number of choice occasions: 12784
Number of parameters estimated: 14
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.571945 |    0.417862 |   -0.24705 |    1.39094
brand_2      |  -0.424738 |    0.332358 |   -1.07615 |    0.22667
brand_3      |  -3.591693 |    0.417917 |   -4.41080 |   -2.77259
brand_4      |  -2.380053 |    0.528097 |   -3.41510 |   -1.34500
f            |   0.401319 |    0.180949 |    0.04666 |    0.75597
p            | -37.462354 |    3.984432 |  -45.27170 |  -29.65301
prev_purch   |   1.285169 |    0.199801 |    0.89357 |    1.67677

Population Covariance M

### Heterogeneity on brand intercepts, diagonal covariance

In [8]:
beta_init = [0.5, -0.5, -3.5, -2, 0.5, -40, 1.5]
sigma_init = np.diag([3, 3, 3, 3])

results_brandhet_diagcov = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=["brand_1", "brand_2", "brand_3", "brand_4", "f", "p", "prev_purch"],
    individual_var="hh",
    n_alts=4,
    eta=eta,
    common_vars=["f", "p", "prev_purch"],
    beta_init=beta_init,
    sigma_init=sigma_init,
    gamma_mask=make_gamma_mask(4, 'diagonal'),
    robust_se=True,
    randomized_starts=1,
    search_rel_width=2,
    search_abs_floor=1,
    opt_method='BFGS',
    output_log=True,
    log_csv_path="output_logs/randcoef_brand_diagcov_log.csv"
)

Randomized starts completed: 1/1 | Elapsed 33.9s | Current LL: -6,480.5839 | Best LL so far: -6,480.5839
Desired error not necessarily achieved due to precision loss.
Converged in 22 iterations.

Log-likelihood: -6480.5839
Number of individuals: 100
Number of choice occasions: 12784
Number of parameters estimated: 11
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.300493 |    0.379581 |   -0.44347 |    1.04446
brand_2      |  -0.375161 |    0.347168 |   -1.05560 |    0.30528
brand_3      |  -3.603469 |    0.414266 |   -4.41542 |   -2.79152
brand_4      |  -3.050881 |    0.588032 |   -4.20340 |   -1.89836
f            |   0.402477 |    0.133018 |    0.14177 |    0.66319
p            | -38.373194 |    3.694417 |  -45.61412 |  -31.13227
prev_purch   |   1.258308 |    0.130437 |    1.00266 |    1.51396


### Heterogeneity on all coefficients, block-diagonal covariance

In [9]:
beta_init = [0.5, -1, -4, -2, 0.5, -40, 1.5]
sigma_init = np.diag([2, 2, 2, 2, 0.5, 40, 0.5])

results_allcoef_blockcov = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    eta=eta,
    common_vars=None,
    beta_init=beta_init,
    sigma_init=sigma_init,
    gamma_mask=make_gamma_mask(7, 'block', [4, 3]),
    robust_se=True,
    randomized_starts=1,
    search_rel_width=2,
    search_abs_floor=1,
    opt_method='BFGS',
    output_log=True,
    log_csv_path="output_logs/randcoef_allcoef_blockcov_log.csv"
)

Randomized starts completed: 1/1 | Elapsed 36.0s | Current LL: -6,429.1934 | Best LL so far: -6,429.1934
Optimization terminated successfully.
Converged in 81 iterations.

Log-likelihood: -6429.1934
Number of individuals: 100
Number of choice occasions: 12784
Number of parameters estimated: 23
Robust standard errors, clustered on the hh variable.

Population Mean of Coefficients
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.251645 |    0.422613 |   -0.57666 |    1.07995
brand_2      |  -0.566034 |    0.344551 |   -1.24134 |    0.10927
brand_3      |  -4.051058 |    0.448544 |   -4.93019 |   -3.17193
brand_4      |  -2.123290 |    0.381129 |   -2.87029 |   -1.37629
f            |   0.390263 |    0.136643 |    0.12245 |    0.65808
p            | -39.180230 |    3.712900 |  -46.45738 |  -31.90308
prev_purch   |   1.788182 |    0.198516 |    1.39910 |    2.17727

Population Covariance M